# Fusion v4: GPU build of the cross-attention multimodal property price model

Changes relative to `Fusion-3-cross-attention.ipynb`:

| # | Change | Why |
|---|---|---|
| 1 | GBDTs run on GPU (XGBoost `device="cuda"`, CatBoost `task_type="GPU"`) | v3 LightGBM ran on CPU |
| 2 | Fusion net: per-epoch progress, per-fold result cache (resume after a crash), bf16 autocast, data resident on GPU | v3 gave no output for hours |
| 3 | Reuses the DINOv2 / BGE caches in `model_artifacts_v3` | no re-embedding |
| 4 | New label-free features: `area_rel_loc`, `area_rel_city_pt`, `proj_freq`; char n-gram SVD of Location+Title and of Name | locality / builder names carry price tier |
| 5 | Hierarchical target encoding (area -> locality -> project -> project+BHK, builder token) | shrinks sparse groups toward their parent |
| 6 | Project-name features are a separate group (`proj`) and ablated | a deployable model (no project name) and a full model are both reported |
| 7 | Hold-out R² split into projects seen / unseen in the dev set | honest generalisation number for the paper |

R² is on `log(price)`. Raw-price R², MAPE, MedAPE and within-20% are reported too. The hold-out test set is never used for tuning.

In [1]:
# !pip install -q xgboost catboost lightgbm transformers sentence-transformers scipy matplotlib
import os
os.environ["USE_TF"] = "0"; os.environ["USE_FLAX"] = "0"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re, json, math, copy, time, random, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, joblib
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.neighbors import BallTree
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

CFG = dict(
    csv="dataset_with_latlon_reindexed.csv",
    image_dir="images_reindexed",
    cache_dir="model_artifacts_v3",   # existing DINOv2 / BGE caches are reused from here
    art_dir="model_artifacts_v4",
    seed=42, test_size=0.15, n_folds=5, inner_folds=5,
    dedupe=True, mad_k=4.0, geo_max_km=80.0,
    use_proj=True,                    # project-name / builder features (not available in a public web form). Ablation reports both.
    consistency_filter=False, consistency_tol=0.20,
    knn_ks=(5, 15, 40),
    img_model="facebook/dinov2-base", img_grid=4, txt_model="BAAI/bge-base-en-v1.5",
    gbdt_trees=8000, run_catboost=True,
    nn_seeds=3, nn_epochs=80, nn_patience=12, nn_bs=256, nn_lr=3e-4, nn_d=256, nn_heads=8, nn_ablation=True,
    allow_cpu=False,
)
SEED = CFG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ART = Path(CFG["art_dir"]); ART.mkdir(exist_ok=True)
CACHE = Path(CFG["cache_dir"]); CACHE.mkdir(exist_ok=True)

dev_t = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", dev_t)
if dev_t.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True; torch.backends.cudnn.benchmark = True
elif not CFG["allow_cpu"]:
    raise RuntimeError("No CUDA GPU visible to PyTorch.")

torch 2.7.1+cu118 | device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | 4.3 GB


In [2]:
# ---------------- 1. Load, parse, clean ----------------
raw = pd.read_csv(CFG["csv"])
raw.columns = raw.columns.str.strip()
raw["img_id"] = np.arange(len(raw))      # reindexed csv row number == image file name
N_ALL = len(raw)
print("raw rows:", N_ALL, "| columns:", list(raw.columns))

# Descriptions may contain price statements (target leakage). Strip them.
CURR = re.compile(
    r"(?:[^\x00-\x7f]+|\brs\.?|\binr)\s*\d[\d.,]*\s*(?:cr(?:ore)?s?|lacs?|lakhs?|l|k)?\b"
    r"|\b\d[\d.,]*\s*(?:cr(?:ore)?s?|lacs?|lakhs?)\b"
    r"|[^.]*\bprice\b[^.]*\.?", re.I)
raw["desc_clean"] = raw["Description"].fillna("").astype(str).map(lambda s: CURR.sub(" ", s).strip())
print(f"descriptions modified by price-scrub: {(raw['desc_clean'] != raw['Description'].fillna('')).mean()*100:.1f}%")

def convert_price(price):
    try:
        s = str(price).lower().replace(",", "")
        v = float(re.sub(r"[^0-9.]", "", s))
        if "cr" in s: return v * 1e7
        if re.search(r"\bl\b|lac|lakh", s): return v * 1e5
        return v
    except Exception:
        return np.nan

d = raw.copy()
d["Price"] = d["Price"].map(convert_price)
n0 = len(d)
d = d[d["Price"].notna() & (d["Price"] >= 1e5)].copy()
print(f"dropped unparsable / < Rs.1 lakh prices: {n0 - len(d)}")

d["BHK"] = d["Property Title"].str.extract(r"(\d+)\s*BHK", flags=re.I)[0].astype(float)
d["Balcony"] = d["Balcony"].map({"Yes": 1, "No": 0}).fillna(0)
if CFG["consistency_filter"]:
    pcol = [c for c in d.columns if c.lower().startswith("price_per")][0]
    ratio = (d["Price"] / pd.to_numeric(d["Total_Area"], errors="coerce")) / pd.to_numeric(d[pcol], errors="coerce")
    ok = ratio.isna() | ((ratio - 1).abs() <= CFG["consistency_tol"])
    print(f"consistency filter: dropped {int((~ok).sum())} rows")
    d = d[ok].copy()
for c in ["BHK", "Baths", "Total_Area", "latitude", "longitude"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
for c in ["BHK", "Baths", "Total_Area"]:
    d[c] = d[c].fillna(d[c].median())
d["BHK"] = d["BHK"].clip(1, 12); d["Baths"] = d["Baths"].clip(1, 12); d["Total_Area"] = d["Total_Area"].clip(lower=100)

def ptype(t):
    t = str(t).lower()
    for key, lab in [("penthouse","penthouse"),("villa","villa"),("independent house","house"),("independent floor","floor"),
                     ("builder floor","floor"),("studio","studio"),("plot","plot"),("land","plot"),("house","house"),
                     ("flat","flat"),("apartment","flat")]:
        if key in t: return lab
    return "other"
d["ptype"] = d["Property Title"].map(ptype)
print(d["ptype"].value_counts().to_dict())

CITY_CENTERS = {"Chennai": (13.0827, 80.2707), "Bangalore": (12.9716, 77.5946), "Mumbai": (19.0760, 72.8777),
                "Pune": (18.5204, 73.8567), "Kolkata": (22.5726, 88.3639), "Delhi": (28.6139, 77.2090)}
ALIASES = {"Chennai": ["chennai"], "Bangalore": ["bangalore", "bengaluru"], "Mumbai": ["mumbai", "thane"],
           "Pune": ["pune"], "Kolkata": ["kolkata", "howrah", "calcutta"],
           "Delhi": ["delhi", "noida", "gurgaon", "gurugram", "ghaziabad", "faridabad"]}
CITY_NAMES = list(CITY_CENTERS)

def haversine_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2-lat1)*p/2)**2 + np.cos(lat1*p)*np.cos(lat2*p)*np.sin((lon2-lon1)*p/2)**2
    return 12742 * np.arcsin(np.sqrt(a))

loc_l = d["Location"].fillna("").str.lower()
text_city = loc_l.map(lambda s: next((c for c, al in ALIASES.items() if any(a in s for a in al)), None))
dm = np.stack([haversine_km(d["latitude"].values, d["longitude"].values, *CITY_CENTERS[c]) for c in CITY_NAMES], 1)
geo_city = pd.Series(np.array(CITY_NAMES)[np.argmin(np.nan_to_num(dm, nan=1e9), 1)], index=d.index)
d["city"] = text_city.fillna(geo_city)
d = d[~(text_city.isna() & d["latitude"].isna())].copy()
d["locality"] = d["Location"].fillna("").str.split(",").str[0].str.strip().str.lower()
d["locality_key"] = d["city"] + "|" + d["locality"]

c_lat = d["city"].map(lambda c: CITY_CENTERS[c][0]); c_lon = d["city"].map(lambda c: CITY_CENTERS[c][1])
own_km = haversine_km(d["latitude"], d["longitude"], c_lat, c_lon)
bad = d["latitude"].isna() | d["longitude"].isna() | (own_km > CFG["geo_max_km"])
med = d[~bad].groupby("locality_key")[["latitude", "longitude"]].median()
for col, cc in [("latitude", c_lat), ("longitude", c_lon)]:
    fill = d["locality_key"].map(med[col]).fillna(cc)
    d.loc[bad, col] = fill[bad]
print(f"geocodes repaired: {int(bad.sum())}")
print(d["city"].value_counts().to_dict())

d["log_area"] = np.log(d["Total_Area"])
d["log_price"] = np.log(d["Price"])
d["ypps"] = d["log_price"] - d["log_area"]          # log price per sqft = learning target

if CFG["dedupe"]:
    before = len(d)
    d = d.drop_duplicates(subset=["Name", "Location", "Total_Area", "Price"], keep="first")
    print(f"exact duplicate listings removed: {before - len(d)}")

g = d.groupby("city")["ypps"]
med_c = g.transform("median")
mad_c = d["ypps"].sub(med_c).abs().groupby(d["city"]).transform("median") * 1.4826
keep = (d["ypps"] - med_c).abs() <= CFG["mad_k"] * mad_c
print(f"outliers removed: {int((~keep).sum())} ({(~keep).mean()*100:.2f}%)")
d = d[keep].copy()
df = d.reset_index(drop=True)

# ---- name / area keys (strings only; label-free)
df["name_l"] = df["Name"].fillna("").astype(str).str.lower().str.strip()
parts = df["Location"].fillna("").str.lower().str.split(",")
df["area2"] = parts.map(lambda p: p[-2].strip() if len(p) >= 2 else p[0].strip())
df["area2_key"] = df["city"] + "|" + df["area2"]
uniq = ("__none__" + df.index.astype(str)).values
df["proj_key"] = np.where(df["name_l"].str.len() >= 3, (df["city"] + "|" + df["name_l"]).values, uniq)
first = df["name_l"].str.extract(r"([a-z]{3,})")[0]
df["name1_key"] = np.where(first.notna(), (df["city"] + "|" + first.fillna("")).values, uniq)
print("final rows:", len(df), "| distinct projects:", df["proj_key"].nunique())

raw rows: 14523 | columns: ['Name', 'Property Title', 'Price', 'Location', 'Total_Area', 'Price_per_SQFT', 'Description', 'Baths', 'Balcony', 'latitude', 'longitude', 'img_id']
descriptions modified by price-scrub: 95.7%
dropped unparsable / < Rs.1 lakh prices: 4
{'flat': 9633, 'house': 4100, 'villa': 736, 'plot': 43, 'studio': 7}
geocodes repaired: 570
{'Bangalore': 4680, 'Pune': 3335, 'Delhi': 2164, 'Chennai': 1590, 'Kolkata': 1392, 'Mumbai': 1358}
exact duplicate listings removed: 370
outliers removed: 166 (1.17%)
final rows: 13983 | distinct projects: 9923


In [3]:
# ---------------- 2. Label-free feature engineering ----------------
df["bath_per_bhk"] = df["Baths"] / (df["BHK"] + 1)
df["log_area_per_bhk"] = df["log_area"] - np.log(df["BHK"])
df["ptype_code"] = df["ptype"].astype("category").cat.codes
PTYPE_CATS = list(df["ptype"].astype("category").cat.categories)
df["city_code"] = df["city"].map({c: i for i, c in enumerate(CITY_NAMES)})

lat0 = df["city"].map(lambda c: CITY_CENTERS[c][0]).values; lon0 = df["city"].map(lambda c: CITY_CENTERS[c][1]).values
df["x_km"] = (df["longitude"].values - lon0) * 111.32 * np.cos(np.radians(lat0))
df["y_km"] = (df["latitude"].values - lat0) * 110.57
df["dist_center"] = np.hypot(df["x_km"], df["y_km"])
df["bearing"] = np.arctan2(df["y_km"], df["x_km"])
for ang in (15, 30, 45):
    a = np.radians(ang)
    df[f"rot{ang}a"] = df["x_km"] * np.cos(a) + df["y_km"] * np.sin(a)
    df[f"rot{ang}b"] = df["y_km"] * np.cos(a) - df["x_km"] * np.sin(a)
df["loc_freq"] = np.log1p(df.groupby("locality_key")["locality_key"].transform("count"))
df["area_rel_loc"] = df["log_area"] - df.groupby("locality_key")["log_area"].transform("median")
df["area_rel_city_pt"] = df["log_area"] - df.groupby(["city", "ptype"])["log_area"].transform("median")
df["proj_freq"] = np.log1p(df.groupby("proj_key")["proj_key"].transform("count"))

KW = {"metro":"metro","school":"school","mall":"mall","hospital":"hospital","park":"park","techpark":"tech park|it park|it hub|software",
      "airport":"airport","highway":"highway|nh\\d|expressway|ring road","gated":"gated|community","pool":"swimming|pool",
      "gym":"gym|fitness|club","luxury":"luxur|premium|elite|lavish","ready":"ready to move|ready-to-move|possession",
      "uc":"under construction|new launch|pre.?launch","furnished":"furnish","parking":"parking","lift":"lift|elevator",
      "vastu":"vastu","corner":"corner"}
dl = df["desc_clean"].str.lower()
for k, p in KW.items():
    df[f"kw_{k}"] = dl.str.contains(p, regex=True).astype(int)
df["desc_len"] = np.log1p(df["desc_clean"].str.len()); df["desc_words"] = np.log1p(df["desc_clean"].str.split().str.len())

GROUPS = {
    "tab": ["log_area","BHK","Baths","Balcony","bath_per_bhk","log_area_per_bhk","ptype_code","city_code","area_rel_loc","area_rel_city_pt","proj_freq"],
    "geo": ["latitude","longitude","x_km","y_km","dist_center","bearing","rot15a","rot15b","rot30a","rot30b","rot45a","rot45b","loc_freq"],
    "kw":  [f"kw_{k}" for k in KW] + ["desc_len", "desc_words"],
}
print({k: len(v) for k, v in GROUPS.items()})

{'tab': 11, 'geo': 13, 'kw': 21}


In [4]:
# ---------------- 3. Frozen embeddings (cached in model_artifacts_v3): DINOv2 image tokens, BGE text embeddings ----------------
from PIL import Image
from torchvision import transforms

def get_image_tokens():
    f_tok, f_has = CACHE / "img_tokens.npy", CACHE / "img_has.npy"
    if f_tok.exists() and f_has.exists():
        return np.load(f_tok), np.load(f_has)
    from transformers import AutoModel
    m = AutoModel.from_pretrained(CFG["img_model"]).to(dev_t).eval()
    if dev_t.type == "cuda": m = m.half()
    tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    g = CFG["img_grid"]; toks = None; has = np.zeros(N_ALL, bool); bs = 64
    with torch.inference_mode():
        for s in range(0, N_ALL, bs):
            ims, ok = [], []
            for i in range(s, min(s + bs, N_ALL)):
                p = os.path.join(CFG["image_dir"], f"{i}.png")
                if os.path.exists(p):
                    ims.append(tf(Image.open(p).convert("RGB"))); ok.append(True)
                else:
                    ims.append(torch.zeros(3, 224, 224)); ok.append(False)
            x = torch.stack(ims).to(dev_t)
            if dev_t.type == "cuda": x = x.half()
            h = m(pixel_values=x).last_hidden_state.float()
            B, _, D = h.shape; side = int(math.sqrt(h.shape[1] - 1))
            patches = h[:, 1:].reshape(B, side, side, D).permute(0, 3, 1, 2)
            pooled = F.adaptive_avg_pool2d(patches, g).flatten(2).transpose(1, 2)
            t = torch.cat([h[:, :1], pooled], 1).cpu().numpy().astype(np.float16)
            if toks is None: toks = np.zeros((N_ALL, t.shape[1], t.shape[2]), np.float16)
            toks[s:s + len(ok)] = t; has[s:s + len(ok)] = ok
            if (s // bs) % 20 == 0: print(f"images {s}/{N_ALL}")
    toks[~has] = 0
    np.save(f_tok, toks); np.save(f_has, has)
    return toks, has

def get_text_embeddings():
    f = CACHE / "txt_emb.npy"
    if f.exists(): return np.load(f)
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(CFG["txt_model"], device=str(dev_t))
    emb = st.encode(raw["desc_clean"].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
    emb = emb.astype(np.float32); np.save(f, emb); return emb

IMG_TOK_ALL, HAS_IMG_ALL = get_image_tokens()
TXT_EMB_ALL = get_text_embeddings()
print("image tokens:", IMG_TOK_ALL.shape, "| images found:", int(HAS_IMG_ALL.sum()), "/", N_ALL, "| text emb:", TXT_EMB_ALL.shape)
assert len(IMG_TOK_ALL) == N_ALL and len(TXT_EMB_ALL) == N_ALL, "cache size does not match the csv"

image tokens: (14523, 17, 768) | images found: 14523 / 14523 | text emb: (14523, 768)


In [5]:
# ---------------- 4. Split, unsupervised reductions, leakage-safe target features ----------------
ids = df["img_id"].values
IMG = IMG_TOK_ALL[ids]; HAS = HAS_IMG_ALL[ids].astype(bool); TXT_EMB = TXT_EMB_ALL[ids]

bins = pd.qcut(df["log_price"], 8, labels=False, duplicates="drop").values
dev_idx, test_idx = train_test_split(np.arange(len(df)), test_size=CFG["test_size"], random_state=SEED, stratify=bins)
skf = StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=SEED)
FOLDS = [(dev_idx[a], dev_idx[b]) for a, b in skf.split(dev_idx, bins[dev_idx])]
print(f"dev={len(dev_idx)} test={len(test_idx)} | folds: {[len(v) for _, v in FOLDS]}")

def tfidf_svd(texts, n, **kw):
    tv = TfidfVectorizer(sublinear_tf=True, **kw).fit(texts[dev_idx])
    k = max(2, min(n, len(tv.vocabulary_) - 1))
    sv = TruncatedSVD(k, random_state=SEED).fit(tv.transform(texts[dev_idx]))
    return sv.transform(tv.transform(texts)).astype(np.float32), tv, sv

desc = df["desc_clean"].fillna("").values
TXT_SVD, tfidf, svd = tfidf_svd(desc, 64, max_features=5000, min_df=3, ngram_range=(1, 2))
loc_text = (df["Location"].fillna("") + " | " + df["Property Title"].fillna("")).str.lower().map(lambda s: CURR.sub(" ", s)).values
LOC_SVD, tv_loc, sv_loc = tfidf_svd(loc_text, 48, analyzer="char_wb", ngram_range=(2, 4), min_df=2, max_features=30000)
NAME_SVD, tv_name, sv_name = tfidf_svd(df["name_l"].values, 32, analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=30000)
N_SVD = TXT_SVD.shape[1]
txt_pca = PCA(32, random_state=SEED).fit(TXT_EMB[dev_idx]); TXT_PCA = txt_pca.transform(TXT_EMB)
img_cls = IMG[:, 0].astype(np.float32)
img_pca = PCA(32, random_state=SEED).fit(img_cls[HAS & np.isin(np.arange(len(df)), dev_idx)]); IMG_PCA = img_pca.transform(img_cls)
IMG_PCA[~HAS] = 0

extra = {}
for j in range(N_SVD): extra[f"svd{j}"] = TXT_SVD[:, j]
for j in range(32): extra[f"tp{j}"] = TXT_PCA[:, j]; extra[f"ip{j}"] = IMG_PCA[:, j]
for j in range(LOC_SVD.shape[1]): extra[f"ls{j}"] = LOC_SVD[:, j]
for j in range(NAME_SVD.shape[1]): extra[f"ns{j}"] = NAME_SVD[:, j]
extra["has_img"] = HAS.astype(int)
base_cols = GROUPS["tab"] + GROUPS["geo"] + GROUPS["kw"]
ALLX = pd.concat([df[base_cols], pd.DataFrame(extra, index=df.index)], axis=1)
GROUPS["text"] = GROUPS["kw"] + [f"svd{j}" for j in range(N_SVD)] + [f"tp{j}" for j in range(32)]
GROUPS["img"] = [f"ip{j}" for j in range(32)] + ["has_img"]
GROUPS["loctxt"] = [f"ls{j}" for j in range(LOC_SVD.shape[1])]
GROUPS["projtxt"] = [f"ns{j}" for j in range(NAME_SVD.shape[1])]

# ---- target features: use ONLY labels of `fit` rows, never of the rows they are computed for.
def target_feats(fit, app):
    y = fit["ypps"].values; out = {}
    tree = BallTree(np.radians(fit[["latitude", "longitude"]].values), metric="haversine")
    kq = min(max(CFG["knn_ks"]), len(fit))
    dist, ind = tree.query(np.radians(app[["latitude", "longitude"]].values), k=kq)
    dkm = dist * 6371.0
    for k in CFG["knn_ks"]:
        kk = min(k, kq); yy = y[ind[:, :kk]]
        out[f"knn{k}_mean"] = yy.mean(1); out[f"knn{k}_std"] = yy.std(1)
        out[f"knn{k}_med"] = np.median(yy, 1); out[f"knn{k}_dist"] = dkm[:, kk - 1]
    k15 = min(15, kq); w = 1.0 / (dkm[:, :k15] + 0.05)
    out["knn_idw"] = (y[ind[:, :k15]] * w).sum(1) / w.sum(1)
    gm = y.mean(); city_mean = fit.groupby("city")["ypps"].mean()
    prior_city = app["city"].map(city_mean).fillna(gm).values

    def te(cols, m, prior, name):
        agg = fit.groupby(cols)["ypps"].agg(["sum", "count"]).reset_index()
        mg = app[cols].reset_index(drop=True).merge(agg, on=cols, how="left")
        s, c = mg["sum"].fillna(0).values, mg["count"].fillna(0).values
        out[f"te_{name}"] = (s + m * prior) / (c + m); out[f"te_{name}_n"] = np.log1p(c)
        return out[f"te_{name}"]
    # hierarchical shrinkage: each level is shrunk toward its parent level
    te(["city", "ptype"], 10, prior_city, "city_pt")
    p_city_pt = out["te_city_pt"]
    te(["city", "ptype", "BHK"], 10, p_city_pt, "city_pt_bhk")
    p_area = te(["area2_key"], 5, prior_city, "area2")
    p_loc = te(["locality_key"], 5, p_area, "loc")
    p_proj = te(["proj_key"], 3, p_loc, "proj")
    te(["proj_key", "BHK"], 3, p_proj, "proj_bhk")
    te(["name1_key"], 5, p_loc, "name1")
    return pd.DataFrame(out, index=app.index)

def oof_target_feats(fit):
    parts = []
    for a, b in KFold(CFG["inner_folds"], shuffle=True, random_state=SEED).split(fit):
        parts.append(target_feats(fit.iloc[a], fit.iloc[b]))
    return pd.concat(parts).loc[fit.index]

t0 = time.time(); TF = []
for tr, va in FOLDS:
    fit = df.iloc[tr]
    TF.append(dict(tr=oof_target_feats(fit), va=target_feats(fit, df.iloc[va]), te=target_feats(fit, df.iloc[test_idx])))
allte = list(TF[0]["tr"].columns)
TF_GROUPS = {"proj": [c for c in allte if c.startswith(("te_proj", "te_name1"))]}
TF_GROUPS["te"] = [c for c in allte if c not in TF_GROUPS["proj"]]
print(f"target features built in {time.time()-t0:.0f}s | te: {len(TF_GROUPS['te'])}  proj: {len(TF_GROUPS['proj'])}")

def assemble(idx, tf, groups):
    cols = [c for g in groups for c in GROUPS.get(g, [])]
    X = ALLX.loc[idx, cols]
    tcols = [c for g in groups for c in TF_GROUPS.get(g, [])]
    if tcols: X = pd.concat([X, tf.loc[idx, tcols]], axis=1)
    return X

Y_LOG = df["log_price"].values; LOGA = df["log_area"].values; YPPS = df["ypps"].values
def metrics(idx, pred_log):
    yt, yp = Y_LOG[idx], np.asarray(pred_log)
    pt, pp = np.exp(yt), np.exp(yp); ape = np.abs(pp - pt) / pt
    return dict(R2_log=r2_score(yt, yp), RMSE_log=np.sqrt(mean_squared_error(yt, yp)), MAE_log=mean_absolute_error(yt, yp),
                R2_price=r2_score(pt, pp), MAPE=ape.mean() * 100, MedAPE=np.median(ape) * 100, Within20=(ape <= 0.20).mean() * 100)

BASE = ["tab", "geo", "loctxt", "te"]
FULL_GROUPS = BASE + ["text", "img"] + (["proj"] if CFG["use_proj"] else [])
print("full model groups:", FULL_GROUPS)

dev=11885 test=2098 | folds: [2377, 2377, 2377, 2377, 2377]
target features built in 6s | te: 21  proj: 6
full model groups: ['tab', 'geo', 'loctxt', 'te', 'text', 'img', 'proj']


In [6]:
# ---------------- 5. XGBoost on GPU: baselines + modality ablation ----------------
import xgboost as xgb
def cv_gbdt(groups, seed=SEED):
    oof = np.full(len(df), np.nan); tp = np.zeros(len(test_idx))
    for f, (tr, va) in enumerate(FOLDS):
        Xtr, Xva, Xte = assemble(tr, TF[f]["tr"], groups), assemble(va, TF[f]["va"], groups), assemble(test_idx, TF[f]["te"], groups)
        m = xgb.XGBRegressor(n_estimators=CFG["gbdt_trees"], learning_rate=0.03, max_depth=7, min_child_weight=3, subsample=0.8,
                             colsample_bytree=0.7, reg_lambda=3.0, tree_method="hist", device="cuda",
                             early_stopping_rounds=300, random_state=seed)
        m.fit(Xtr, YPPS[tr], eval_set=[(Xva, YPPS[va])], verbose=False)
        oof[va] = m.predict(Xva) + LOGA[va]; tp += (m.predict(Xte) + LOGA[test_idx]) / len(FOLDS)
    return oof, tp

ABL = [("tab", ["tab"]), ("tab+geo", ["tab", "geo"]), ("tab+geo+loctxt", ["tab", "geo", "loctxt"]),
       ("tab+geo+loctxt+TE", BASE), ("+text", BASE + ["text"]), ("+img", BASE + ["img"]),
       ("deployable (no project name): +text+img", BASE + ["text", "img"]),
       ("+proj only", BASE + ["proj"]), ("FULL: +text+img+proj", BASE + ["text", "img", "proj"])]
rows, OOF, TEST = [], {}, {}
for name, groups in ABL:
    t0 = time.time(); oof, tp = cv_gbdt(groups)
    r = dict(config=name, **{f"cv_{k}": v for k, v in metrics(dev_idx, oof[dev_idx]).items() if k in ("R2_log", "MAPE")},
             **{f"test_{k}": v for k, v in metrics(test_idx, tp).items() if k in ("R2_log", "MAPE")})
    rows.append(r); print(f"{name:42s} cv R2={r['cv_R2_log']:.4f}  test R2={r['test_R2_log']:.4f}  ({time.time()-t0:.0f}s)", flush=True)
    if set(groups) == set(FULL_GROUPS): OOF["xgb"], TEST["xgb"] = oof, tp
ablation_gbdt = pd.DataFrame(rows); ablation_gbdt.to_csv(ART / "ablation_xgb.csv", index=False)
ablation_gbdt.round(4)

tab                                        cv R2=0.6800  test R2=0.6785  (13s)
tab+geo                                    cv R2=0.7822  test R2=0.7789  (27s)
tab+geo+loctxt                             cv R2=0.7781  test R2=0.7811  (31s)
tab+geo+loctxt+TE                          cv R2=0.7822  test R2=0.7869  (39s)
+text                                      cv R2=0.7833  test R2=0.7887  (87s)
+img                                       cv R2=0.7827  test R2=0.7861  (45s)
deployable (no project name): +text+img    cv R2=0.7831  test R2=0.7901  (100s)
+proj only                                 cv R2=0.7830  test R2=0.7882  (37s)
FULL: +text+img+proj                       cv R2=0.7853  test R2=0.7913  (90s)


,config,cv_R2_log,cv_MAPE,test_R2_log,test_MAPE
0,tab,0.6800,40.5742,0.6785,39.0289
1,tab+geo,0.7822,31.5615,0.7789,30.4161
2,tab+geo+loctxt,0.7781,32.0009,0.7811,30.3193
3,tab+geo+loctxt+TE,0.7822,31.6106,0.7869,29.7827
4,+text,0.7833,31.6858,0.7887,30.1265
5,+img,0.7827,31.6137,0.7861,29.7735
6,deployable (no project name): +text+img,0.7831,31.7203,0.7901,29.9712
7,+proj only,0.7830,31.5362,0.7882,29.6676
8,FULL: +text+img+proj,0.7853,31.4736,0.7913,29.8215


In [7]:
# ---------------- 6. CatBoost on GPU (native categoricals) ----------------
if CFG["run_catboost"]:
    from catboost import CatBoostRegressor
    cats = ["city", "ptype", "locality_key", "area2_key"] + (["proj_key", "name1_key"] if CFG["use_proj"] else [])
    def build(task):
        kw = dict(task_type="GPU", devices="0") if task == "GPU" else dict(thread_count=-1)
        return CatBoostRegressor(iterations=6000, learning_rate=0.05, depth=8, l2_leaf_reg=5, loss_function="RMSE",
                                 early_stopping_rounds=300, random_seed=SEED, verbose=0, **kw)
    oof = np.full(len(df), np.nan); tp = np.zeros(len(test_idx))
    for f, (tr, va) in enumerate(FOLDS):
        mk = lambda idx, tf: pd.concat([assemble(idx, tf, FULL_GROUPS), df.loc[idx, cats].astype(str)], axis=1)
        Xtr, Xva, Xte = mk(tr, TF[f]["tr"]), mk(va, TF[f]["va"]), mk(test_idx, TF[f]["te"])
        try:
            m = build("GPU"); m.fit(Xtr, YPPS[tr], eval_set=(Xva, YPPS[va]), cat_features=cats)
        except Exception as ex:
            print("catboost GPU failed, using CPU:", str(ex)[:150]); m = build("CPU"); m.fit(Xtr, YPPS[tr], eval_set=(Xva, YPPS[va]), cat_features=cats)
        oof[va] = m.predict(Xva) + LOGA[va]; tp += (m.predict(Xte) + LOGA[test_idx]) / len(FOLDS)
        print(f"catboost fold {f}: R2_log={r2_score(Y_LOG[va], oof[va]):.4f}", flush=True)
    OOF["catboost"], TEST["catboost"] = oof, tp
    print("catboost  cv:", {k: round(v, 4) for k, v in metrics(dev_idx, oof[dev_idx]).items()})
    print("catboost test:", {k: round(v, 4) for k, v in metrics(test_idx, tp).items()})

catboost fold 0: R2_log=0.8017
catboost fold 1: R2_log=0.7871
catboost fold 2: R2_log=0.7714
catboost fold 3: R2_log=0.7791
catboost fold 4: R2_log=0.7875
catboost  cv: {'R2_log': 0.7856, 'RMSE_log': np.float64(0.3983), 'MAE_log': 0.2856, 'R2_price': 0.5912, 'MAPE': np.float64(31.5518), 'MedAPE': np.float64(20.6833), 'Within20': np.float64(48.7421)}
catboost test: {'R2_log': 0.7918, 'RMSE_log': np.float64(0.3878), 'MAE_log': 0.2796, 'R2_price': 0.6114, 'MAPE': np.float64(29.8724), 'MedAPE': np.float64(19.9201), 'Within20': np.float64(50.1907)}


In [8]:
# ---------------- 6b. Diagnostics: where is the error coming from? ----------------
pcol = [c for c in raw.columns if c.lower().startswith("price_per")]
if pcol:
    pg = pd.to_numeric(raw.loc[df["img_id"].values, pcol[0]].values, errors="coerce")
    ratio = pd.Series((df["Price"].values / df["Total_Area"].values) / pg)
    print(f"Price/Area vs listed {pcol[0]}: ratio percentiles")
    print(ratio.describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())
    print("rows differing by >20%:", round(((ratio - 1).abs() > .2).mean() * 100, 1), "%")
e = df.iloc[dev_idx].assign(pred=OOF["xgb"][dev_idx]); e["err"] = e["pred"] - e["log_price"]
for key in ["city", "ptype"]:
    print(f"\nCV error by {key}")
    print(e.groupby(key).apply(lambda g: pd.Series({"n": len(g), "R2": r2_score(g["log_price"], g["pred"]) if len(g) > 2 else np.nan,
                                                   "RMSE_log": np.sqrt((g["err"] ** 2).mean())})).round(3))
print("\n25 largest CV errors")
print(e.loc[e["err"].abs().sort_values(ascending=False).index].head(25)[["Name", "Location", "BHK", "Total_Area", "Price", "err"]].to_string())

Price/Area vs listed Price_per_SQFT: ratio percentiles
count    13983.000
mean         1.042
std          2.049
min          0.001
5%           0.999
25%          1.000
50%          1.000
75%          1.000
95%          1.001
max        100.204
rows differing by >20%: 0.1 %

CV error by city
                n     R2  RMSE_log
city                              
Bangalore  3740.0  0.723     0.377
Chennai    1316.0  0.803     0.340
Delhi      1776.0  0.679     0.597
Kolkata    1143.0  0.779     0.345
Mumbai     1147.0  0.846     0.379
Pune       2763.0  0.845     0.313

CV error by ptype
             n     R2  RMSE_log
ptype                          
flat    7971.0  0.836     0.310
house   3292.0  0.662     0.549
plot      31.0  0.831     0.264
studio     6.0  0.081     0.730
villa    585.0  0.767     0.460

25 largest CV errors
                                                       Name                                              Location   BHK  Total_Area        Price       err
13702  

In [9]:
# ---------------- 7. Cross-attention fusion network (GPU, bf16, all data resident on GPU) ----------------
USE_AMP = dev_t.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
print("mixed precision:", AMP_DTYPE if USE_AMP else "off")

NN_NUM = [c for c in GROUPS["tab"] if c not in ("ptype_code", "city_code")] + GROUPS["geo"] + GROUPS["kw"]
TF_NN = TF_GROUPS["te"] + (TF_GROUPS["proj"] if CFG["use_proj"] else [])
BLK_COLS = GROUPS["loctxt"] + (GROUPS["projtxt"] if CFG["use_proj"] else [])

def vocab(col, minc):
    vc = df.iloc[dev_idx][col].value_counts(); v = {k: i + 1 for i, k in enumerate(vc[vc >= minc].index)}
    return v, df[col].map(v).fillna(0).astype(int).values
LOC_VOCAB, LOC_IDX = vocab("locality_key", 3)
AR_VOCAB, AR_IDX = vocab("area2_key", 3)
cat_cols = [df["city_code"].values + 1, df["ptype_code"].values + 1, LOC_IDX, AR_IDX]
CARDS = [len(CITY_NAMES) + 1, len(PTYPE_CATS) + 1, len(LOC_VOCAB) + 1, len(AR_VOCAB) + 1]
if CFG["use_proj"]:
    PR_VOCAB, PR_IDX = vocab("proj_key", 3); cat_cols.append(PR_IDX); CARDS.append(len(PR_VOCAB) + 1)
CAT_ALL = np.stack(cat_cols, 1).astype(np.int64)

blk_sc = StandardScaler().fit(ALLX.loc[dev_idx, BLK_COLS])
BLK_ALL = np.nan_to_num(np.clip(blk_sc.transform(ALLX[BLK_COLS]), -6, 6)).astype(np.float32)

IMG_G = torch.as_tensor(IMG).to(dev_t)                                   # fp16 [N,17,768]
T1_G = torch.as_tensor(TXT_EMB, dtype=torch.float32).to(dev_t)
T2_G = torch.as_tensor(TXT_SVD, dtype=torch.float32).to(dev_t)
BLK_G = torch.as_tensor(BLK_ALL).to(dev_t)
HAS_G = torch.as_tensor(HAS).to(dev_t)
CAT_G = torch.as_tensor(CAT_ALL).to(dev_t)

class XBlock(nn.Module):
    def __init__(s, d, h, p):
        super().__init__()
        s.nq, s.nk, s.nf = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        s.attn = nn.MultiheadAttention(d, h, dropout=p, batch_first=True)
        s.ff = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(p), nn.Linear(2 * d, d)); s.dr = nn.Dropout(p)
    def forward(s, q, kv, kv_pad=None):
        k = s.nk(kv); a, _ = s.attn(s.nq(q), k, k, key_padding_mask=kv_pad, need_weights=False)
        q = q + s.dr(a); return q + s.dr(s.ff(s.nf(q)))

class FusionLayer(nn.Module):
    def __init__(s, d, h, p):
        super().__init__()
        s.t_ctx, s.i_t, s.x_t = XBlock(d, h, p), XBlock(d, h, p), XBlock(d, h, p)
        s.joint = nn.TransformerEncoderLayer(d, h, 2 * d, p, activation="gelu", batch_first=True, norm_first=True)
    def forward(s, T, I, X, i_pad, x_pad):
        T = s.t_ctx(T, torch.cat([I, X], 1), torch.cat([i_pad, x_pad], 1))
        I, X = s.i_t(I, T), s.x_t(X, T)
        pad = torch.cat([torch.zeros(T.shape[:2], dtype=torch.bool, device=T.device), i_pad, x_pad], 1)
        S = s.joint(torch.cat([T, I, X], 1), src_key_padding_mask=pad)
        nT, nI = T.size(1), I.size(1)
        return S[:, :nT], S[:, nT:nT + nI], S[:, nT + nI:]

class FusionNet(nn.Module):
    def __init__(s, n_num, cards, img_dim, n_img_tok, txt_dim, svd_dim, blk_dim, d=256, h=8, n_tab=3, n_img=2, n_fuse=4, p=0.15):
        super().__init__()
        enc = lambda: nn.TransformerEncoderLayer(d, h, 2 * d, p, activation="gelu", batch_first=True, norm_first=True)
        s.num_w = nn.Parameter(torch.randn(n_num, d) * 0.02); s.num_b = nn.Parameter(torch.zeros(n_num, d))
        s.cat_emb = nn.ModuleList([nn.Embedding(c, d) for c in cards]); s.cls = nn.Parameter(torch.zeros(1, 1, d))
        s.blk = nn.Sequential(nn.LayerNorm(blk_dim), nn.Linear(blk_dim, d))
        s.tab_enc = nn.TransformerEncoder(enc(), n_tab, enable_nested_tensor=False)
        s.img_proj = nn.Sequential(nn.LayerNorm(img_dim), nn.Linear(img_dim, d)); s.img_pos = nn.Parameter(torch.randn(1, n_img_tok, d) * 0.02)
        s.img_enc = nn.TransformerEncoder(enc(), n_img, enable_nested_tensor=False)
        s.t1 = nn.Sequential(nn.LayerNorm(txt_dim), nn.Linear(txt_dim, d)); s.t2 = nn.Sequential(nn.LayerNorm(svd_dim), nn.Linear(svd_dim, d))
        s.null = nn.Parameter(torch.zeros(1, 1, d)); s.txt_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        s.fuse = nn.ModuleList([FusionLayer(d, h, p) for _ in range(n_fuse)]); s.norm = nn.LayerNorm(d)
        s.head = nn.Sequential(nn.Linear(2 * d, 512), nn.GELU(), nn.Dropout(p), nn.Linear(512, 128), nn.GELU(), nn.Linear(128, 1))
        s.skip = nn.Linear(n_num, 1)
    def forward(s, xn, xc, img, t1, t2, blk, has_i, has_t):
        B = xn.size(0)
        tab = ([s.cls.expand(B, -1, -1), xn.unsqueeze(-1) * s.num_w + s.num_b]
               + [e(xc[:, i]).unsqueeze(1) for i, e in enumerate(s.cat_emb)] + [s.blk(blk).unsqueeze(1)])
        T = s.tab_enc(torch.cat(tab, 1))
        I = s.img_enc(s.img_proj(img) + s.img_pos)
        X = torch.cat([s.null.expand(B, -1, -1), s.t1(t1).unsqueeze(1), s.t2(t2).unsqueeze(1)], 1) + s.txt_pos
        i_pad = (~has_i).unsqueeze(1).expand(B, I.size(1))
        x_pad = torch.stack([torch.zeros_like(has_t), ~has_t, ~has_t], 1)
        for layer in s.fuse: T, I, X = layer(T, I, X, i_pad, x_pad)
        T = s.norm(T)
        return s.head(torch.cat([T[:, 0], T[:, 1:].mean(1)], 1)).squeeze(1) + s.skip(xn).squeeze(1)

def make_tensors(idx, Xnum):
    return dict(num=torch.as_tensor(Xnum, dtype=torch.float32).to(dev_t), idx=torch.as_tensor(idx, dtype=torch.long).to(dev_t))

def get_batch(T, b):
    ix = T["idx"][b]
    return T["num"][b], CAT_G[ix], IMG_G[ix].float(), T1_G[ix], T2_G[ix], BLK_G[ix], HAS_G[ix]

PREP = []
for f, (tr, va) in enumerate(FOLDS):
    asm = lambda idx, tf: pd.concat([ALLX.loc[idx, NN_NUM], tf.loc[idx, TF_NN]], axis=1).values.astype(np.float64)
    Xtr, Xva, Xte = asm(tr, TF[f]["tr"]), asm(va, TF[f]["va"]), asm(test_idx, TF[f]["te"])
    sc = StandardScaler().fit(Xtr)
    fx = lambda X: np.nan_to_num(np.clip(sc.transform(X), -6, 6))
    mu, sd = YPPS[tr].mean(), YPPS[tr].std()
    PREP.append(dict(tr=make_tensors(tr, fx(Xtr)), va=make_tensors(va, fx(Xva)), te=make_tensors(test_idx, fx(Xte)),
                     ytr=torch.tensor((YPPS[tr] - mu) / sd, dtype=torch.float32, device=dev_t),
                     yva=torch.tensor((YPPS[va] - mu) / sd, dtype=torch.float32, device=dev_t), mu=mu, sd=sd, n_num=Xtr.shape[1]))
print("NN numeric features:", PREP[0]["n_num"], "| block dim:", BLK_ALL.shape[1], "| image tokens:", IMG.shape[1:], "| cards:", CARDS)

@torch.no_grad()
def nn_predict(model, T, use_img, use_txt, bs=1024):
    model.eval(); out = []
    for i in range(0, T["num"].size(0), bs):
        xn, xc, im, t1, t2, bl, hs = get_batch(T, slice(i, i + bs)); n = xn.size(0)
        hi = hs if use_img else torch.zeros(n, dtype=torch.bool, device=dev_t)
        ht = torch.full((n,), bool(use_txt), dtype=torch.bool, device=dev_t)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            out.append(model(xn, xc, im, t1, t2, bl, hi, ht).float())
    return torch.cat(out)

def run_nn(f, seed, use_img=True, use_txt=True, p_drop_mod=0.15, verbose=True):
    t_start = time.time(); torch.manual_seed(seed); np.random.seed(seed)
    D = PREP[f]; Ttr = D["tr"]; n = Ttr["num"].size(0); bs = CFG["nn_bs"]
    model = FusionNet(D["n_num"], CARDS, IMG.shape[2], IMG.shape[1], TXT_EMB.shape[1], TXT_SVD.shape[1], BLK_ALL.shape[1],
                      d=CFG["nn_d"], h=CFG["nn_heads"]).to(dev_t)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["nn_lr"], weight_decay=1e-2)
    spe = math.ceil(n / bs); total = CFG["nn_epochs"] * spe
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (3 * spe)) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    loss_fn = nn.HuberLoss(delta=1.5); best, best_state, bad = 1e9, None, 0
    for ep in range(CFG["nn_epochs"]):
        model.train(); perm = torch.randperm(n, device=dev_t)
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            if len(b) < 16: continue
            xn, xc, im, t1, t2, bl, hs = get_batch(Ttr, b)
            hi = (hs & (torch.rand(len(b), device=dev_t) > p_drop_mod)) if use_img else torch.zeros(len(b), dtype=torch.bool, device=dev_t)
            ht = (torch.rand(len(b), device=dev_t) > p_drop_mod) if use_txt else torch.zeros(len(b), dtype=torch.bool, device=dev_t)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                pred = model(xn, xc, im, t1, t2, bl, hi, ht)
            loss = loss_fn(pred.float(), D["ytr"][b])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        v = F.mse_loss(nn_predict(model, D["va"], use_img, use_txt), D["yva"]).item()
        if ep == 0: print(f"    epoch 1 took {time.time()-t_start:.1f}s (upper bound for this run: {(time.time()-t_start)*CFG['nn_epochs']/60:.1f} min)", flush=True)
        if v < best - 1e-4: best, bad, best_state = v, 0, {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else: bad += 1
        if verbose and ep % 5 == 0: print(f"    ep{ep:03d} val_mse(z)={v:.4f} best={best:.4f} [{time.time()-t_start:.0f}s]", flush=True)
        if bad >= CFG["nn_patience"]: break
    model.load_state_dict(best_state)
    pv = nn_predict(model, D["va"], use_img, use_txt).cpu().numpy() * D["sd"] + D["mu"]
    pt = nn_predict(model, D["te"], use_img, use_txt).cpu().numpy() * D["sd"] + D["mu"]
    print(f"    run fold{f} seed{seed}: {ep+1} epochs, best val mse(z)={best:.4f}, {time.time()-t_start:.0f}s", flush=True)
    del model, opt; torch.cuda.empty_cache()
    return pv, pt, best_state

RUN_ID = hashlib.md5(json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()[:8]
def cv_nn(use_img=True, use_txt=True, seeds=None, tag="nn", save=False):
    seeds = seeds or CFG["nn_seeds"]; oof = np.full(len(df), np.nan); tp = np.zeros(len(test_idx))
    for f, (tr, va) in enumerate(FOLDS):
        fp = ART / f"nncache_{RUN_ID}_{tag}_s{seeds}_f{f}.npz"
        if fp.exists():
            z = np.load(fp); pv_m, pt_m = z["pv"], z["pt"]; print(f"[{tag}] fold {f}: loaded from cache", flush=True)
        else:
            pvs, pts = [], []
            for s in range(seeds):
                pv, pt, st = run_nn(f, SEED + 100 * f + s, use_img, use_txt, verbose=(s == 0)); pvs.append(pv); pts.append(pt)
                if save: torch.save(st, ART / f"nn_{tag}_fold{f}_seed{s}.pt")
            pv_m, pt_m = np.mean(pvs, 0), np.mean(pts, 0); np.savez(fp, pv=pv_m, pt=pt_m)
        oof[va] = pv_m + LOGA[va]; tp += (pt_m + LOGA[test_idx]) / len(FOLDS)
        print(f"[{tag}] fold {f}: R2_log={r2_score(Y_LOG[va], oof[va]):.4f}", flush=True)
    return oof, tp

mixed precision: torch.bfloat16
NN numeric features: 70 | block dim: 80 | image tokens: (17, 768) | cards: [7, 6, 1100, 1086, 630]


In [10]:
# ---------------- 7b. Speed probe: ONE run before the full loop ----------------
t0 = time.time()
pv, pt, _ = run_nn(0, SEED, verbose=True)
va0 = FOLDS[0][1]
print(f"one run: {time.time()-t0:.0f}s | fold-0 R2_log = {r2_score(Y_LOG[va0], pv + LOGA[va0]):.4f}")
print(f"estimated full training: ~{(time.time()-t0)*CFG['n_folds']*CFG['nn_seeds']/60:.0f} min")

    epoch 1 took 32.3s (upper bound for this run: 43.1 min)
    ep000 val_mse(z)=0.8651 best=0.8651 [32s]
    ep005 val_mse(z)=0.5257 best=0.5257 [181s]
    ep010 val_mse(z)=0.5421 best=0.4760 [330s]
    ep015 val_mse(z)=0.5250 best=0.4760 [473s]
    ep020 val_mse(z)=0.5754 best=0.4744 [608s]
    ep025 val_mse(z)=0.5206 best=0.4744 [743s]
    run fold0 seed42: 30 epochs, best val mse(z)=0.4744, 877s
one run: 877s | fold-0 R2_log = 0.7738
estimated full training: ~219 min


In [11]:
# ---------------- 8. Train the fusion network (5 folds x seeds). Finished folds are cached: re-running resumes. ----------------
t0 = time.time()
OOF["nn"], TEST["nn"] = cv_nn(True, True, tag="fusion", save=True)
print(f"done in {(time.time()-t0)/60:.1f} min")
print("NN cv  :", {k: round(v, 4) for k, v in metrics(dev_idx, OOF["nn"][dev_idx]).items()})
print("NN test:", {k: round(v, 4) for k, v in metrics(test_idx, TEST["nn"]).items()})

[fusion] fold 0: loaded from cache
[fusion] fold 0: R2_log=0.7992
[fusion] fold 1: loaded from cache
[fusion] fold 1: R2_log=0.7931
[fusion] fold 2: loaded from cache
[fusion] fold 2: R2_log=0.7771
[fusion] fold 3: loaded from cache
[fusion] fold 3: R2_log=0.7878
[fusion] fold 4: loaded from cache
[fusion] fold 4: R2_log=0.7938
done in 0.0 min
NN cv  : {'R2_log': 0.7903, 'RMSE_log': np.float64(0.3938), 'MAE_log': 0.2818, 'R2_price': 0.588, 'MAPE': np.float64(31.3365), 'MedAPE': np.float64(20.3295), 'Within20': np.float64(49.2385)}
NN test: {'R2_log': 0.8007, 'RMSE_log': np.float64(0.3794), 'MAE_log': 0.2693, 'R2_price': 0.6468, 'MAPE': np.float64(28.9813), 'MedAPE': np.float64(19.2318), 'Within20': np.float64(51.5253)}


In [ ]:
# ---------------- 9. NN modality ablation (1 seed each; for the paper) ----------------
abl_nn = []
if CFG["nn_ablation"]:
    for tag, ui, ut in [("tab_only", False, False), ("plus_text", False, True), ("plus_image", True, False)]:
        o, t = cv_nn(ui, ut, seeds=1, tag=tag)
        abl_nn.append(dict(config=tag, cv_R2_log=r2_score(Y_LOG[dev_idx], o[dev_idx]), test_R2_log=r2_score(Y_LOG[test_idx], t)))
    abl_nn.append(dict(config=f"plus_text_and_image (full, {CFG['nn_seeds']} seeds)", cv_R2_log=r2_score(Y_LOG[dev_idx], OOF["nn"][dev_idx]),
                       test_R2_log=r2_score(Y_LOG[test_idx], TEST["nn"])))
    abl_nn = pd.DataFrame(abl_nn); abl_nn.to_csv(ART / "ablation_nn.csv", index=False)
abl_nn if len(abl_nn) else "skipped"

[tab_only] fold 0: loaded from cache
[tab_only] fold 0: R2_log=0.7702
[tab_only] fold 1: loaded from cache
[tab_only] fold 1: R2_log=0.7669
[tab_only] fold 2: loaded from cache
[tab_only] fold 2: R2_log=0.7454
[tab_only] fold 3: loaded from cache
[tab_only] fold 3: R2_log=0.7588
[tab_only] fold 4: loaded from cache
[tab_only] fold 4: R2_log=0.7637
[plus_text] fold 0: loaded from cache
[plus_text] fold 0: R2_log=0.7710
[plus_text] fold 1: loaded from cache
[plus_text] fold 1: R2_log=0.7788
[plus_text] fold 2: loaded from cache
[plus_text] fold 2: R2_log=0.7557
[plus_text] fold 3: loaded from cache
[plus_text] fold 3: R2_log=0.7708


In [ ]:
# ---------------- 10. Blend, final report, bootstrap CI, seen/unseen projects ----------------
from scipy.optimize import nnls
names = [n for n in ["xgb", "catboost", "nn"] if n in OOF]
A = np.stack([OOF[n][dev_idx] - LOGA[dev_idx] for n in names], 1)
w, _ = nnls(A, YPPS[dev_idx]); w = w / w.sum()
OOF["blend"] = sum(wi * OOF[n] for wi, n in zip(w, names)); TEST["blend"] = sum(wi * TEST[n] for wi, n in zip(w, names))
print("blend weights:", dict(zip(names, w.round(3))))

rep = []
for n in names + ["blend"]:
    rep.append(dict(model=n, split="CV (dev OOF)", **metrics(dev_idx, OOF[n][dev_idx])))
    rep.append(dict(model=n, split="HOLD-OUT TEST", **metrics(test_idx, TEST[n])))
report = pd.DataFrame(rep); report.to_csv(ART / "final_report.csv", index=False)
print(report.round(4).to_string(index=False))

rng = np.random.default_rng(SEED); yt, yp = Y_LOG[test_idx], TEST["blend"]
boots = [r2_score(yt[i], yp[i]) for i in (rng.integers(0, len(yt), len(yt)) for _ in range(1000))]
r2_test = r2_score(yt, yp)
print(f"\nBlend hold-out test R2(log price) = {r2_test:.4f} | 95% bootstrap CI [{np.percentile(boots, 2.5):.4f}, {np.percentile(boots, 97.5):.4f}]")
print("TARGET R2 >= 0.85:", "REACHED" if r2_test >= 0.85 else "NOT reached; see ablation and per-city table")

tdf = df.iloc[test_idx].assign(pred=TEST["blend"], true=yt)
print("\nHold-out R2 by city:")
print(tdf.groupby("city").apply(lambda g: pd.Series({"n": len(g), "R2_log": r2_score(g["true"], g["pred"]) if len(g) > 2 else np.nan,
                                                  "MedAPE%": np.median(np.abs(np.exp(g["pred"]) - np.exp(g["true"])) / np.exp(g["true"])) * 100})).round(3))

dev_keys = set(df.iloc[dev_idx]["proj_key"])
pk = df.iloc[test_idx]["proj_key"]
seen = (pk.isin(dev_keys) & ~pk.str.startswith("__none__")).values
print("\nHold-out R2 split by whether the project name appears in the dev set:")
for lab, msk in [("project seen in dev", seen), ("project unseen / unnamed", ~seen)]:
    if msk.sum() > 2: print(f"  {lab:26s} n={int(msk.sum()):5d}  R2_log={r2_score(yt[msk], yp[msk]):.4f}")

In [ ]:
# ---------------- 11. Plots + artifacts ----------------
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(yt, yp, s=6, alpha=.4); lo, hi = yt.min(), yt.max(); ax[0].plot([lo, hi], [lo, hi], "r--")
ax[0].set(xlabel="true log price", ylabel="predicted log price", title=f"Hold-out test, R2={r2_test:.3f}")
res = yp - yt; ax[1].hist(res, bins=60); ax[1].set(title="Residual (log price)", xlabel="pred - true")
plt.tight_layout(); plt.savefig(ART / "test_scatter_residuals.png", dpi=160); plt.show()

joblib.dump(dict(cfg=CFG, dev_idx=dev_idx, test_idx=test_idx, blend_weights=dict(zip(names, w)), features=GROUPS, tf_groups=TF_GROUPS,
                 ptype_cats=PTYPE_CATS, city_centers=CITY_CENTERS, loc_vocab=LOC_VOCAB, tfidf=tfidf, svd=svd, txt_pca=txt_pca, img_pca=img_pca,
                 tv_loc=tv_loc, sv_loc=sv_loc, tv_name=tv_name, sv_name=sv_name), ART / "pipeline_meta.joblib")
np.savez(ART / "oof_test_preds.npz", **{f"oof_{k}": v for k, v in OOF.items()}, **{f"test_{k}": v for k, v in TEST.items()})
df.iloc[test_idx].assign(pred_log=TEST["blend"]).to_csv(ART / "test_predictions.csv", index=False)
print("saved to", ART.resolve())